# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**
- Hiểu cấu trúc và chất lượng dữ liệu.
- Phát hiện xu hướng, tính mùa vụ và các mẫu đặc trưng.
- Cung cấp insight để định hướng feature engineering ở notebook 02.

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
from google.colab import drive

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing

In [ ]:
# Cấu hình
warnings.filterwarnings('ignore') # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option('display.float_format', '{:.2f}'.format) # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update({
    'figure.dpi'         : 150,
    'axes.grid'          : True,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'font.family'        : 'DejaVu Sans',
}) # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    'Good'                            : '#00E400',
    'Moderate'                        : '#FFFF00',
    'Unhealthy for Sensitive Groups'  : '#FF7E00',
    'Unhealthy'                       : '#FF0000',
    'Very Unhealthy'                  : '#8F3F97',
    'Hazardous'                       : '#7E0023',
}

AQI_BINS   = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}

# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:    return 'Good'
    elif value <= 100: return 'Moderate'
    elif value <= 150: return 'Unhealthy for Sensitive Groups'
    elif value <= 200: return 'Unhealthy'
    elif value <= 300: return 'Very Unhealthy'
    else:              return 'Hazardous'

# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [val / MAX_AQI for val in AQI_BINS] # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]] # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(zip(positions, gradient_colors)) # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list('AQI_gradient', color_mapping) # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# Cái này để cho nhóm trưởng lưu Files nha (Mặc kệ nó đi)
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

## **01. Khám Phá Dữ Liệu (EDA)**

#### **Cấu trúc notebook:**
1. Đọc File dữ liệu và nhận xét sơ lược.
2. Thống kê mô tả, đánh giá và kiểm tra dữ liệu.
3. Phân phối US AQI.
4. Chuỗi thời gian.
5. Tính mùa vụ.
6. Time Series Decomposition.
7. So sánh AQI theo năm 2022-2026.
8. Mối tương quan.
9. Bản đồ Folium TPHCM.
10. Tổng kết.

### **1.1. Đọc File dữ liệu và nhận xét sơ lược**

In [ ]:
folder_ID  = "1_5C0jQ9fI_ALET0DJPypVJeKS4Cxjtt7"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output='data', quiet=False)

df_city_info              = pd.read_csv('data/city_info.csv')
df_data_dictionary        = pd.read_csv('data/data_dictionary.csv')
df_air_quality_historical = pd.read_csv('data/air_quality_historical.csv')

In [ ]:
df_city_info

> File này bao gồm Metadata thành phố: tọa độ, dân số, khu vực hành chính.

In [ ]:
df_data_dictionary

> File này mô tả chi tiết từng cột, đơn vị và ý nghĩa.

In [ ]:
df_air_quality_historical

> File dữ liệu chính gồm chuỗi thời gian (ngày) - các chỉ số ô nhiễm và AQI.

#### **Tổng quan dữ liệu:**
Dữ liệu về chất lượng không khí của thành phố Hồ Chí Minh gồm 1298 dòng (01/08/2022 - 18/02/2026) và 12 cột (các chỉ số đánh giá chất lượng không khí và ngày đánh giá).

### **1.2. Thống kê mô tả, đánh giá và kiểm tra dữ liệu**

In [ ]:
df_air_quality_historical.describe().round(3)

In [ ]:
info_df = pd.DataFrame(
    {
        "Kiểu dữ liệu": df_air_quality_historical.dtypes,
        "Giá trị thiếu": df_air_quality_historical.isnull().sum(),
        "Tỷ lệ thiếu (%)": (df_air_quality_historical.isnull().sum() / len(df_air_quality_historical) * 100).round(3),
    }
)
print(info_df.to_string())

#### **Đánh giá dữ liệu chính:**

- **Dữ liệu bị thiếu (Missing Values)**

**Hầu hết các cột đều có 1295 dòng dữ liệu.** Tuy nhiên, us_aqi và european_aqi chỉ có 1294 dòng. Điều này có nghĩa là có 1 dòng bị thiếu (NaN) ở hai cột này. Tổng quan lại bộ dữ liệu có 1298 dòng nghĩa là có 3-4 dòng bị thiếu (NaN) ở các cột chỉ số đánh giá chất lượng không khí.

**Tỷ lệ missing values rất thấp (~0.2-0.3%),** chủ yếu ở 4 ngày đầu tháng 08/2022. Dữ liệu sạch và đầy đủ - rất thuận lợi cho việc xây dựng mô hình.

---

- **Chất lượng không khí tổng thể (Thông qua US AQI hoặc European AQI)**

**Mức độ trung bình:** Chỉ số trung bình của US AQI là 81.56 và trung vị (50%) là 78.75. Điều này cho thấy phần lớn thời gian, chất lượng không khí nằm ở mức Trung bình (Moderate). Không khí có thể chấp nhận được nhưng những người cực kỳ nhạy cảm nên hạn chế hoạt động ngoài trời lâu.

**Mức độ nguy hiểm:** Chỉ số tối đã của US AQI lên tới 158.17. Theo chuẩn Mỹ, AQI > 150 là mức Kém (Unhealthy), ảnh hưởng xấu đến sức khỏe của tất cả mọi người. Độ lệch chuẩn (Std = 22.42) cho thấy thỉnh thoảng TP.HCM có những đợt ô nhiễm khá nặng.

### **1.3. Phân phối US AQI**


In [ ]:
# Chuyển đổi cột 'date' sang định dạng datetime và trích xuất ngày, tháng, năm
df_air_quality_historical['date']  = pd.to_datetime(df_air_quality_historical['date'])
df_air_quality_historical['day']   = df_air_quality_historical['date'].dt.day
df_air_quality_historical['month'] = df_air_quality_historical['date'].dt.month
df_air_quality_historical['year']  = df_air_quality_historical['date'].dt.year

In [ ]:
# Vẽ histogram với màu theo từng ngưỡng AQI
aqi_data = df_air_quality_historical['us_aqi'].dropna()
fig, ax = plt.subplots(figsize=(12, 6))

# Tính bins thủ công để gán màu từng cột
n_bins = 30
counts, bin_edges = np.histogram(aqi_data, bins=n_bins)

for i in range(len(counts)):
    bin_mid   = (bin_edges[i] + bin_edges[i + 1]) / 2
    bin_width = bin_edges[i + 1] - bin_edges[i]
    normalized_val = bin_mid / MAX_AQI
    bar_color = AQI_gradient_cmap(normalized_val)

    # Màu chữ cho viền
    edge_color = 'gray'

    ax.bar(
        bin_edges[i],
        counts[i],
        width=bin_width,
        align='edge',
        color=bar_color,
        edgecolor=edge_color,
        linewidth=0.5,
        alpha=0.92,
    )

# Đường KDE
kde_x = np.linspace(aqi_data.min(), aqi_data.max(), 300)
kde   = gaussian_kde(aqi_data, bw_method='scott')
# Scale KDE về cùng trục tần suất (Count)
kde_y = kde(kde_x) * len(aqi_data) * (aqi_data.max() - aqi_data.min()) / n_bins
ax.plot(kde_x, kde_y, color='#1a1a2e', linewidth=2.2, label='KDE')

# Vẽ đường phân cách ngưỡng AQI
for threshold, label in zip(AQI_BINS[1:-1], AQI_LABELS[1:]):
    if threshold <= aqi_data.max():
        ax.axvline(threshold, color='#333333', linestyle='--', linewidth=0.8, alpha=0.6)

# Legend màu AQI
patches = [
    mpatches.Patch(
        facecolor=AQI_COLORS[lbl],
        edgecolor='#666666',
        label=lbl
    )
    for lbl in AQI_LABELS
    if aqi_data.max() >= AQI_BINS[AQI_LABELS.index(lbl)]
]
ax.legend(
    handles=patches,
    title='Mức AQI (US EPA)',
    loc='upper right',
    fontsize=9,
    title_fontsize=10,
    framealpha=0.9,
)

ax.set_title('Phân Phối Biến US AQI', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('US AQI', fontsize=11)
ax.set_ylabel('Tần suất', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
aqi_data = df_air_quality_historical['us_aqi'].dropna().sort_values().reset_index(drop=True)
total_samples = len(aqi_data)


# Tính toán số lượng và % để đưa vào Chú thích
aqi_categories = aqi_data.apply(AQI_CATEGORY)
aqi_counts = aqi_categories.value_counts().reindex(AQI_LABELS).fillna(0)

legend_labels = []
for label in AQI_LABELS:
    count = aqi_counts[label]
    pct = (count / total_samples) * 100
    legend_labels.append(f'{label} ({pct:.2f}%)')

N = 360
indices = np.linspace(0, total_samples - 1, N).astype(int)
smooth_aqi = aqi_data.iloc[indices].values
slice_colors = [AQI_gradient_cmap(val / MAX_AQI) for val in smooth_aqi]

# Vẽ biểu đồ tròn
fig, ax = plt.subplots(figsize=(10, 6))
wedges, _ = ax.pie(
  [1] * N,
  colors=slice_colors,
  startangle=90,
  counterclock=True,
  wedgeprops={'linewidth': 0, 'edgecolor': 'none'}
  )

# Chuyển đổi toàn bộ các lát cắt thành tên danh mục tương ứng
smooth_categories = [AQI_CATEGORY(val) for val in smooth_aqi]

for i in range(N - 1):
  # Nếu lát cắt hiện tại và lát cắt kế tiếp khác nhóm nhau sẽ vẽ ranh giới
  if smooth_categories[i] != smooth_categories[i + 1]:
    # Tính góc độ của đường ranh giới (mỗi lát tương ứng 1 độ, xuất phát từ góc 90)
    angle_deg = 90 + (i + 1)
    angle_rad = np.radians(angle_deg)

    # Tính tọa độ điểm x, y trên rìa đường tròn (Bán kính R = 1)
    x = np.cos(angle_rad)
    y = np.sin(angle_rad)

    # Vẽ đường phân cách nét liền từ tâm (0,0) ra rìa (x,y)
    ax.plot(
        [0, x], [0, y],
        color='#2c3e50',
        linestyle='-',
        linewidth=0.5,
        alpha=0.9,
        zorder=3
        )

outer_border = plt.Circle(
    (0, 0), 1.0,
    edgecolor='#2c3e50',
    fill=False,
    linewidth=0.5,
    zorder=3
    )
ax.add_artist(outer_border)

legend_patches = [
    mpatches.Patch(facecolor=AQI_COLORS[lbl], edgecolor='#666666', label=legend_labels[i])
    for i, lbl in enumerate(AQI_LABELS)
    if aqi_counts[lbl] > 0
    ]

ax.legend(
    handles=legend_patches,
    title='Mức độ US AQI và Tỷ lệ',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=11,
    framealpha=0.9
    )
ax.set_title('Phân Phối Mức Độ US AQI', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

####  **Phân tích Phân phối biến US AQI**

- **Xu hướng trung tâm (Central Tendency)**

**Mode:** ~70-85 (nằm trong ngưỡng Moderate)

**Phân phối lệch:** Lệch phải

**Ước lượng Mean & Median:** Mean > Median ~75-85 do đuôi phải kéo dài

---

- **Độ phân tán**

**Khoảng biến thiên:** Khoảng từ 10 đến hơn 160

**Vùng tập trung chính (IQR ước lượng):** 60-100 (phần lớn Moderate)

**Nhận xét độ lệch chuẩn:** khá lớn (~25-35), dữ liệu trải rộng

---

- **Hình dạng phân phối**

**Dạng tổng quát:** Lệch phải, một đỉnh

**Độ lệch (Skewness):** Dương → phần lớn ngày có AQI thấp/trung bình, nhưng tồn tại những ngày ô nhiễm nặng kéo skewness lên cao

**Độ nhọn (Kurtosis):** Đuôi béo hơn chuẩn (leptokurtic) nên xác suất xuất hiện giá trị cực cao không thể bỏ qua

---

- **Điểm ngoại lai (Outliers)**

**Outlier:** AQI > 150, rải rác đến hơn 160

**Tần suất / Tỷ lệ outlier:** ~1-2% (Unhealthy + Very Unhealthy + Hazardous gộp lại)

**Nguyên nhân có thể:** Ô nhiễm mùa khô, cháy rừng, giao thông cao điểm

**Xử lý:** Giữ lại để phân tích rủi ro, tách nhóm riêng khi xây dựng mô hình

---

- **So sánh với Phân phối Lý thuyết**

**Đường KDE so với Bell Curve:** KDE lệch phải, đỉnh cao hơn và đuôi phải dài hơn Bell Curve chuẩn

**Kiểm định trực quan (Q-Q plot nếu có):** Chưa có - nên vẽ bổ sung để xác nhận

**Kết luận:** **không thể** giả định phân phối chuẩn, phù hợp hơn với phân phối Log-normal hoặc Gamma

---

- **Kết luận theo Ngữ cảnh**

**Phân bố theo ngưỡng AQI:**

  | Mức                            | Khoảng    | Tỷ lệ    |
  |--------------------------------|-----------|----------|
  | Good                           | (0,50]    | ~4.2%    |
  | Moderate                       | (50,100]  | ~76.5%   |
  | Unhealthy for Sensitive Groups | (10,150]  | ~18.6%   |
  | Unhealthy                      | (150,200] | ~0.7%    |
  | Very Unhealthy                 | (200,300] | ~0.0%    |
  | Hazardous                      | (300,)    | ~0.0%    |

**Nhận xét tổng thể:** Chất lượng không khí phần lớn ở mức Moderate (76.5%) - chưa tốt nhưng chưa nguy hiểm. Đáng lo ngại là gần 1/5 số ngày (18.6%) ảnh hưởng đến nhóm nhạy cảm (trẻ em, người già, người có bệnh về hô hấp).



### **1.4. Chuỗi thời gian**

In [ ]:
# Đưa cột 'date' làm index và thiết lập tần suất hàng ngày (D)
df_freq = df_air_quality_historical.set_index('date')['us_aqi'].asfreq('D')

# Xử lý dữ liệu bị thiếu bằng cách điền bù tuyến tính (interpolate) và fill ngược (bfill)
df_freq = df_freq.interpolate(method='linear').bfill()

df_rolling = pd.DataFrame({'Daily AQI': df_freq})
# Tính trung bình động (Rolling Mean) cho 7 ngày và 30 ngày
df_rolling['7-day MA']  = df_freq.rolling(window=7, min_periods=1).mean()
df_rolling['30-day MA'] = df_freq.rolling(window=30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 7))

# Vẽ đường AQI gốc (Hàng ngày)
ax.plot(df_rolling.index, df_rolling['Daily AQI'], label='AQI Hàng ngày', color='#FF9800', linewidth=1, alpha=0.5)

# Vẽ đường Rolling Mean 7 ngày
ax.plot(df_rolling.index, df_rolling['7-day MA'], label='Trung bình động 7 ngày', color='#4CAF50', linewidth=1.5, alpha=0.9)
# Vẽ đường Rolling Mean 30 ngày (Xu hướng dài hạn)
ax.plot(df_rolling.index, df_rolling['30-day MA'], label='Trung bình động 30 ngày', color='#F44336', linewidth=2.5)

# Định dạng các thành phần của biểu đồ
ax.set_title('Biến Động US AQI Và Các Đường Trung Bình Động (Rolling Mean)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Thời gian', fontsize=11)
ax.set_ylabel('Chỉ số US AQI', fontsize=11)

ax.legend(loc='lower right', frameon=True, fontsize=11, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()

#### **Phân tích Rolling Mean**

- **Đường màu cam nhạt (AQI Hàng ngày):** Góc nhìn Vi mô & Độ nhiễu (Micro & Noise)

**Sự biến động dữ dội:** Dữ liệu gốc biến thiên cực kỳ mạnh. Trong cùng một tháng, chỉ số có thể rớt xuống mức rất thấp (~20: Good) nhưng ngay lập tức vọt lên mức cao (>140: Unhealthy for Sensitive Groups/Unhealthy).

> **Giá trị:** Đường này cho biết rằng chất lượng không khí phụ thuộc vào các yếu tố có tính "tức thời" (ví dụ: Một cơn mưa lớn làm sạch bụi, hoặc một ngày lặng gió khiến khói bụi tích tụ).

> **Hạn chế:** Nếu chỉ nhìn đường này, sẽ không thể xác định được xu hướng chung vì có quá nhiều nhiễu.

---

- **Đường màu xanh lá (Trung bình động 7 ngày):** Xu hướng ngắn hạn (Short-term Trends)

**Các đợt ô nhiễm/trong lành:** Đường 7 ngày đã gọt bớt các đỉnh quá nhọn của dữ liệu ngày. Nó cho thấy các đợt không khí tốt hay xấu thường kéo dài thành từng chuỗi vài ngày đến một tuần, chứ không chỉ chớp nhoáng 1-2 ngày.

> **Biên độ dao động:** Mức độ dao động của đường xanh hẹp hơn đường cam, chủ yếu nằm trong khoảng 40 đến 120. Các đỉnh của đường xanh chính là những tuần lễ mà chất lượng không khí ở mức đáng báo động.

> **Giá trị:** Phù hợp để phân tích tác động của các hình thái thời tiết theo tuần (như một đợt gió mùa, một đợt nắng nóng kéo dài) lên chỉ số AQI.

---

- **Đường màu đỏ (Trung bình động 30 ngày):** Góc nhìn Vĩ mô, Mùa vụ & Bất thường (Macro, Seasonality & Anomalies)

**Tính mùa vụ:** Nhìn dọc theo trục thời gian, có thể thấy rõ các vùng trũng (chỉ số AQI thấp - không khí tốt) thường rơi vào khoảng giữa năm (quanh mốc tháng 7 của 2023, 2024). Ngược lại, những đỉnh (AQI cao - ô nhiễm) thường tụ vào cuối năm hoặc đầu năm mới.

**Xu hướng vĩ mô:** Từ giữa 2022 đến giữa 2024, mức độ ô nhiễm cao nhất (đỉnh màu đỏ) dao động dưới mức 100. Dữ liệu có tính lặp lại khá ổn định.

> **Phát hiện Bất thường:** Từ cuối năm 2024 kéo dài đến giữa năm 2025, đường màu đỏ tăng vọt và tạo một đỉnh kỷ lục (vượt ngưỡng 120, tiệm cận 140). Khối lượng vùng dưới đồ thị (diện tích ô nhiễm) ở giai đoạn này lớn và kéo dài bất thường so với các năm trước. Sau đó, đến đầu 2026, chỉ số mới bắt đầu hạ nhiệt.

### **1.5. Tính mùa vụ**

#### **Boxplot**

In [ ]:
sns.set_theme(style="whitegrid")
# Định nghĩa tên các tháng để hiển thị trực quan trên trục x
month_labels = ['Tháng 1', 'Tháng 2', 'Tháng 3', 'Tháng 4', 'Tháng 5', 'Tháng 6',
                'Tháng 7', 'Tháng 8', 'Tháng 9', 'Tháng 10', 'Tháng 11', 'Tháng 12']

# Thiết lập giao diện
fig, ax = plt.subplots(figsize=(10, 6))

# Vẽ Boxplot cho trung bình US AQI của 5 năm
sns.boxplot(x='month', y='us_aqi', data=df_air_quality_historical, color='#e0e0e0', ax=ax, width=0.6)

# Vẽ đường xu hướng cho US AQI qua từng năm
df_monthly = df_air_quality_historical.groupby(['year', 'month'])['us_aqi'].mean().reset_index()
sns.lineplot(x=df_monthly['month'] - 1, y='us_aqi', hue='year', data=df_monthly, marker='o', linewidth=2.5, palette='tab10', ax=ax)

ax.set_title('Phân Phối Chỉ Số US AQI Theo Từng Tháng Trong Năm', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('Tháng', fontsize=12)
ax.set_ylabel('US AQI', fontsize=12)

ax.set_xticks(range(12))
ax.set_xticklabels(month_labels, rotation=45)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.4), ncol=5, framealpha=0.0)


plt.tight_layout()
plt.show()

#### **Phân tích Boxplot**

- **Chu kỳ chung:**

**Khoảng thời gian có chất lượng không khí kém nhất:** Đầu năm, cuối năm và giữa năm (Tháng 1, Tháng 6, Tháng 11, Tháng 12).

**Khoảng thời gian có chất lượng không khí tốt nhất:** Mùa xuân (Tháng 2, Tháng 3, Tháng 4). Biểu đồ cho thấy sự suy giảm rõ rệt của chỉ số AQI trong giai đoạn này.

---

- **Mức độ biến động và các ngày báo động (Outliers):**

**Outliers xuất hiện rất nhiều ở phía trên các boxplot,** đặc biệt dày đặc vào các tháng 1, 5, 8, 9, và 11. Điều này cho thấy mặc dù chất lượng không khí có thể theo chu kỳ, nhưng vẫn thường xuyên xuất hiện những ngày có mức độ ô nhiễm tăng đột biến và bất thường trong các tháng này.

**Tháng 12 có một điểm ngoại lệ nằm rất thấp (dưới mức 20),** cho thấy thỉnh thoảng vẫn có những ngày không khí cực kỳ trong lành xen kẽ trong tháng cao điểm ô nhiễm.

---

- **Sự khác biệt theo từng năm:** Mặc dù có tính mùa vụ chung, nhưng diễn biến cụ thể của từng năm lại có sự khác biệt đáng chú ý.

**Năm 2025 (Đường màu đỏ):** Là năm ghi nhận mức độ ô nhiễm nghiêm trọng nhất so với các năm khác trong đa số các tháng. Đặc biệt, năm này tạo ra một đỉnh cực kỳ cao vào Tháng 6, gây ảnh hướng đến xu hướng ổn định của các năm trước đó.

**Năm 2024 (Đường màu xanh lá):** Thể hiện rõ nhất chu kỳ sạch của không khí vào mùa xuân khi chạm đáy thấp nhất vào Tháng 4, nhưng sau đó lại tăng dần và duy trì ở mức trung bình cao vào nửa cuối năm.

**Năm 2023 (Đường màu cam):** Có xu hướng khá bình ổn trong nửa cuối năm, ít có những pha dao động mạnh như năm 2025.

> **Tóm lại:** Chất lượng không khí mang tính mùa vụ rõ ràng với hai giai đoạn ô nhiễm chính là mùa Đông (Tháng 1, 11, 12) và đầu mùa Hè (Tháng 6), trong khi mùa Xuân (Tháng 2 đến Tháng 4) thường là thời điểm không khí trong lành nhất.

#### **Heatmap**

In [ ]:
# Lệnh unstack() giúp xoay dữ liệu thành dạng ma trận: hàng là Tháng, cột là Ngày
df_daily = df_air_quality_historical.groupby(['month', 'day'])['us_aqi'].mean().unstack()
fig, ax = plt.subplots(figsize=(15, 8))

# Vẽ heatmap với dải màu thể hiện mức độ ô nhiễm tăng dần
sns.heatmap(df_daily, cmap=AQI_gradient_cmap, vmin=0, vmax=500, ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Chỉ số US AQI (Chuẩn EPA)'})
ax.grid(False)

ax.set_title('Mức Độ Ô Nhiễm (US AQI) Theo Ngày và Tháng', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('Ngày', fontsize=12)
ax.set_ylabel('Tháng', fontsize=12)

ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

#### **Phân tích Heatmap**

- **Vùng ô nhiễm cao:** Tập trung rõ ràng vào các tháng
  T11, T12, T1, T2, T6 - xác nhận tính mùa vụ từ phân tích boxplot.

- **Vùng không khí sạch:** Rải rác tháng T4-T6,
  phù hợp với mùa mưa bắt đầu ở TPHCM.

- **Không có pattern theo ngày trong tuần rõ ràng:** Các ngày không cho thấy sự khác biệt đáng kể, nghĩa là ô nhiễm phụ thuộc vào tháng (mùa) nhiều hơn ngày cụ thể → feature `month` quan trọng hơn `day`.

### **1.6. Time Series Decomposition**

**Định nghĩa Time Series Decomposition (Phân rã chuỗi thời gian)**: Là một kỹ thuật được sử dụng để phân tách một chuỗi thời gian thành ba thành phần chính - xu hướng (trend), tính thời vụ (seasonality) và residuals (nhiễu hay phần dư). Quá trình này giúp hiểu rõ các quy luật tiềm ẩn trong dữ liệu và đóng vai trò cực kỳ quan trọng đối với các tác vụ như dự báo và phát hiện điểm bất thường.

In [ ]:
# Biến đổi date làm index và khai báo tuần suất là hàng ngày (D)
df_freq = df_air_quality_historical.set_index('date')['us_aqi'].asfreq('D')

# Xử lý dữ liệu bị thiếu bằng cách điền bù
df_freq = df_freq.interpolate(method='linear').bfill()

# Thực hiện Decomposition
result = seasonal_decompose(
    df_freq,
    model  = 'additive',   # AQI = Trend + Seasonal + Residual
    period = 365           # Chu kỳ 1 năm
)
# model = 'multiplicative': AQI = Trend × Seasonal × Residual

# Vẽ 4 Subplot cho: Dữ liệu gốc (Observed), Trend, Seasonal và Residual
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Time Series Decomposition - US AQI TP.HCM', fontsize=14, fontweight='bold')

# Observed
result.observed.plot(ax=axes[0], color='#2196F3', linewidth=0.8)
axes[0].set_ylabel('Observed')
axes[0].set_title('Chuỗi gốc')

# Trend
result.trend.plot(ax=axes[1], color='#FF5722', linewidth=1.8)
axes[1].set_ylabel('Trend')
axes[1].set_title('Xu hướng (Dài hạn)')

# Seasonal
result.seasonal.plot(ax=axes[2], color='#4CAF50', linewidth=0.8)
axes[2].set_ylabel('Seasonal'); axes[2].set_title('Mùa vụ (Chu kỳ 365 ngày)')

# Residual = Observed - (Trend + Seasonal)
# Nếu thành phần Trend và Seasonal giải thích được 100% sự biến động của không khí, thì Residual sẽ bằng đúng 0.
result.resid.plot(ax=axes[3], color='#9C27B0', linewidth=0.8)
axes[3].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Ngày')
axes[3].set_title('Phần dư (Nhiễu ngẫu nhiên)')

plt.tight_layout()
plt.show()

#### **Phân tích Time Series Decomposition**

- **Trend (Xu hướng dài hạn):** AQI không đi theo một đường thẳng mà chia làm 2 giai đoạn rõ rệt:

**Năm 2023:** Có xu hướng giảm nhẹ (từ mức ~80 xuống đáy ~70).

**Từ đầu 2024 đến giữa 2025:** Xu hướng tăng mạnh và liên tục (từ đáy ~70 tăng mạnh lên mức đỉnh >90).

> **Kết luận:** Nhìn chung, xu hướng chính trong thời gian gần đây là tăng, cho thấy chất lượng không khí cơ bản tại TP.HCM đang có dấu hiệu xấu dần.

---

- **Seasonal (Tính mùa vụ):** Chu kỳ mùa vụ thể hiện rất rõ ràng và đều đặn:

**AQI tăng cao vào các tháng đầu năm và cuối năm** (tương ứng mùa khô, ít mưa, bụi không được rửa trôi và hiện tượng nghịch nhiệt).

**AQI giảm thấp vào các tháng giữa năm** (tương ứng mùa mưa).

**Biên độ dao động:** Dựa vào trục y, yếu tố mùa vụ làm dao động AQI trong khoảng từ -30 đến +45 đơn vị. (Nghĩa là vào đỉnh điểm mùa khô, riêng yếu tố thời tiết đã cộng thêm tới ~45 đơn vị vào chỉ số ô nhiễm chung).

---

- **Residual (Phần dư/Nhiễu ngẫu nhiên):**

Phần dư dao động mạnh quanh mức 0.

Có rất nhiều gai nhọn bất thường vượt mốc $|residual| > 20$, đặc biệt có những ngày nhiễu đột biến chạm ngưỡng $\pm 40$.

> **Nguyên nhân & Lưu ý:** Điều này phản ánh các sự kiện ô nhiễm cục bộ (kẹt xe nghiêm trọng, cháy nổ, đốt rác...) hoặc những đợt mưa bão rửa trôi bụi đột ngột. Đây là những biến động ngẫu nhiên không có tính chu kỳ → Cần đặc biệt lưu ý khi đưa vào huấn luyện mô hình vì nhiễu lớn sẽ làm giảm độ chính xác của các mô hình dự báo tương lai.

### **1.7. So sánh AQI theo năm 2022-2026**

In [ ]:
# Tạo Figure và Axes
fig, ax = plt.subplots(figsize=(12, 6))

# Nhóm dữ liệu theo năm và tháng, tính trung bình AQI
df_monthly = df_air_quality_historical.groupby(['year', 'month'])['us_aqi'].mean().reset_index()

sns.lineplot(
    data=df_monthly,
    x='month',
    y='us_aqi',
    hue='year',
    palette='tab10',
    marker='o',
    markersize=8,
    linewidth=2.5,
    ax=ax
)

ax.set_title('So Sánh Xu Hướng US AQI Các Năm (2022-2026)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tháng', fontsize=11)
ax.set_ylabel('US AQI', fontsize=11)

month_labels = ['Tháng 1', 'Tháng 2', 'Tháng 3', 'Tháng 4', 'Tháng 5', 'Tháng 6',
                'Tháng 7', 'Tháng 8', 'Tháng 9', 'Tháng 10', 'Tháng 11', 'Tháng 12']
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, rotation=0)
ax.legend(title='Năm', loc='upper right', frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()

#### **Phân tích US AQI trung bình qua từng năm**

- **Tính mùa vụ rõ rệt của chất lượng không khí**: Nhìn vào xu hướng chung của các đường (đặc biệt là 2023, 2024 và đầu 2025), nhận thấy chất lượng không khí có tính chu kỳ theo mùa:

**Tháng ô nhiễm nhất:** Thường rơi vào đầu năm (Tháng 1) và cuối năm (Tháng 11, Tháng 12). AQI trong các tháng này thường xuyên vượt mức 80, cá biệt năm 2025 lên tới gần 110 vào tháng 1. Điều này thường do hiện tượng nghịch nhiệt vào mùa đông (giữ lại bụi bẩn ở tầng thấp) hoặc do các hoạt động gia tăng vào dịp cuối năm.

**Tháng không khí sạch nhất:** Thường rơi vào giai đoạn mùa xuân (Tháng 2 đến Tháng 4). Hầu hết các đường đều có xu hướng đi xuống tạo thành hình đáy "chữ V" hoặc "chữ U" trong giai đoạn này, điểm thấp nhất thường loanh quanh mức 50-75.

---

- **Sự bất thường và kỷ lục ô nhiễm của năm 2025:** Năm 2025 là một năm đáng báo động nhất trong toàn bộ chuỗi dữ liệu:

**Mức nền cao hơn hẳn:** Đường màu đỏ gần như nằm trên tất cả các đường còn lại trong suốt cả năm, chứng tỏ chất lượng không khí trung bình của năm 2025 tệ hơn hẳn so với giai đoạn 2022-2024.

**Đỉnh điểm Tháng 6 cực đoan:** Điểm nổi bật nhất của toàn bộ biểu đồ là AQI vào tháng 6/2025, vọt lên mức hơn 130 (mức màu cam/đỏ theo chuẩn US AQI, có hại cho sức khỏe nhóm nhạy cảm). Trong khi ở các năm khác, tháng 6 chỉ dao động quanh mức 80. Sự gia tăng đột biến này chắc chắn xuất phát từ một sự kiện bất thường (ví dụ: Đợt nắng nóng cực đoan gây ra ozone tăng vọt, cháy rừng nghiêm trọng, hoặc sự cố khí tượng diện rộng).

---

- **Năm 2024 - Điểm sáng về không khí sạch:** Trái ngược với 2025, năm 2024 có vẻ là một năm trong lành nhất:

Trong 4 tháng đầu năm, chỉ số AQI của 2024 liên tục giảm sâu và chạm đáy khoảng 50 vào Tháng 4 (mức thấp nhất của toàn bộ đồ thị). Đây là mức không khí được đánh giá là Tốt (Good).

Mặc dù có tăng lên vào giữa và cuối năm, đường 2024 nhìn chung vẫn duy trì ở nửa dưới của đồ thị, cho thấy các điều kiện khí tượng hoặc chính sách kiểm soát môi trường trong năm này có thể đã phát huy tác dụng tốt.

---

- **Tháng 6 là thời điểm có biên độ dao động lớn nhất:** Nếu như ở tháng 3, tháng 4, mức độ chênh lệch AQI giữa các năm khá hẹp (khoảng từ 50 đến 75), thì đến tháng 6, sự phân hóa diễn ra cực kỳ mạnh mẽ.

Khoảng cách từ mức thấp nhất (khoảng 80 của năm 2024) đến mức cao nhất (hơn 130 của năm 2025) là rất lớn. Điều này cho thấy thời tiết hoặc các yếu tố tác động đến môi trường vào tháng 6 hàng năm là rất khó lường và thiếu tính ổn định.

---

- **Ghi chú về tính toàn vẹn của dữ liệu:** Từ biểu đồ, cũng có thể thấy tình trạng thu thập dữ liệu:

**Năm 2022:** Thiếu hụt dữ liệu từ tháng 1 đến tháng 7, vì chỉ bắt đầu ghi nhận từ tháng 8 trở đi.

**Năm 2026:** Vì hiện tại mới là năm 2026, đồ thị chỉ mới có điểm dữ liệu cho Tháng 1 (gần 90) và Tháng 2 (khoảng 75). Tuy nhiên, có thể thấy xu hướng đầu năm 2026 đang bám khá sát với chu kỳ giảm của năm 2023 và 2025, dự báo tháng 3 và tháng 4 sắp tới không khí có thể sẽ tiếp tục tốt lên.

### **1.8. Mối tương quan**

In [ ]:
# Tạo vùng vẽ với kích thước lớn để chứa đủ các biến
fig, ax = plt.subplots(figsize=(14, 10))

# 1. Lọc ra các cột chứa dữ liệu dạng số (loại bỏ cột ngày tháng dạng string/datetime)
df_numeric = df_air_quality_historical.select_dtypes(include=[np.number])

# Vì trong df_air_quality_historical đã tạo các cột year, month, day, nên loại bỏ chúng ra khỏi ma trận
# vì tương quan tuyến tính của thời gian thường không thể hiện rõ qua Pearson
cols_to_drop = ['year', 'month', 'day']
df_numeric = df_numeric.drop(columns=[col for col in cols_to_drop if col in df_numeric.columns], errors='ignore')

# 2. Tính toán ma trận tương quan Pearson
corr_matrix = df_numeric.corr()


# 3. Vẽ Heatmap
# Sử dụng dải màu 'coolwarm' (Xanh - Đỏ): Xanh là tương quan âm, Đỏ là tương quan dương
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    vmax=1.0,
    vmin=-1.0,
    center=0,
    annot=True,       # Hiển thị con số bên trong ô
    fmt='.2f',        # Làm tròn 2 chữ số thập phân
    square=True,
    linewidths=.5,
    cbar_kws={"shrink": .7, 'label': 'Hệ số tương quan (Pearson)'},
    ax=ax
)

ax.grid(False)
ax.set_title('Ma Trận Tương Quan Giữa Các Biến Số Khí Tượng Và Ô Nhiễm', fontsize=14, fontweight='bold', pad=15)

# Xoay nhãn trục X và Y để chữ không bị đè lên nhau
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

#### **Phân tích Mối tương quan giữa các biến**

- **Dấu hiệu của nguồn ô nhiễm từ Giao Thông đô thị**

**Dữ liệu từ biểu đồ:** nitrogen_dioxide (NO2) và carbon_monoxide (CO) có độ tương quan dương khá mạnh khoảng 0.69.

> **Nhận xét:** Trong môi trường đô thị, CO và NO2 thường là 2 chất sinh ra từ quá trình đốt cháy nhiên liệu không hoàn toàn của động cơ đốt trong (đặc biệt là xe máy, ô tô).

> Sự đồng biến mạnh mẽ này cho thấy nguyên nhân cốt lõi gây ô
nhiễm không khí trong tập dữ liệu này rất có thể đến từ hoạt động giao thông, chứ không hẳn là từ các hiện tượng tự nhiên như cháy rừng hay bão cát.

---

- **Ảnh hưởng của bụi mịn lên thang đo AQI**

**Dữ liệu từ biểu đồ:** us_aqi có mức tương quan cực cao với pm2_5 (0.92) và pm10 (0.80). Trong khi đó, tương quan giữa AQI với các khí độc khác như ozone (0.22) hay sulphur_dioxide (0.32) lại khá yếu.

> **Nhận xét:** Mặc dù AQI là chỉ số tổng hợp của nhiều loại khí thải, nhưng ở TPHCM, bụi mịn PM2.5 chính là yếu tố chi phối hoàn toàn quyết định bầu không khí có độc hại hay không. Khí độc (như SO2, O3) có tồn tại nhưng hiếm khi vượt ngưỡng đủ để làm thay đổi AQI bằng bụi mịn.

---

- **Hiệu ứng "Làm sạch" và "Pha loãng" của thời tiết (UV Index)**

**Dữ liệu từ biểu đồ:** uv_index có tương quan âm với hầu hết các chất ô nhiễm: NO2 (-0.47), CO (-0.30), PM2.5 (-0.28), và cả US_AQI (-0.27).

> **Nhận xét:** Những ngày có chỉ số tia UV cao thường là những ngày nắng gắt, bầu trời quang đãng. Nhiệt độ bề mặt cao tạo ra các luồng đối lưu nhiệt mạnh (không khí bốc lên cao), giúp khuếch tán và thổi bay các chất ô nhiễm lên tầng khí quyển cao hơn, làm sạch không khí sát mặt đất. Ngược lại, vào những ngày âm u (UV thấp), hiện tượng nghịch nhiệt thường xảy ra, nhốt các chất ô nhiễm ở lại dưới thấp khiến chúng ta hít phải nhiều hơn.

---

- **Bí ẩn của Phản ứng Quang hóa học (Photochemical Smog)**

**Dữ liệu từ biểu đồ:** ozone có tương quan dương với uv_index (0.31) nhưng lại có tương quan âm với nitrogen_dioxide (-0.23).

> **Nhận xét:** Khác với các chất khác, Ozone ở tầng thấp hiếm khi được phát thải trực tiếp từ ống xả. Nó được hình thành khi tia cực tím (UV) chiếu vào phản ứng với các tiền chất như NO2. Do đó:
    
1. Khi nắng càng gắt (UV tăng), Ozone sinh ra càng nhiều (tương quan dương 0.31).

2. Khi NO2 tham gia vào phản ứng hóa học để tạo ra Ozone, bản thân lượng NO2 sẽ bị tiêu hao dần (tương quan âm -0.23). Đây là một bằng chứng rõ rệt cho sự hình thành khói mù quang hóa trong khu vực.

---

- **Khả năng giám sát ô nhiễm từ Vệ Tinh**

**Dữ liệu từ biểu đồ:** aerosol_optical_depth (AOD - Độ sâu quang học Aerosol) có tương quan khá tốt với pm10 (0.49) và pm2_5 (0.47).

> **Nhận xét:** AOD không phải là thông số đo dưới mặt đất mà là chỉ số đo bằng ảnh vệ tinh từ trên không gian (thể hiện mức độ ánh sáng bị chặn lại bởi các hạt lơ lửng). Mức tương quan dương ~0.5 cho thấy các nhà nghiên cứu hoàn toàn có thể dùng dữ liệu vệ tinh (AOD) để ước lượng xấp xỉ mức độ ô nhiễm bụi mịn (PM) ở mặt đất cho những vùng sâu vùng xa, nơi không có ngân sách để lắp đặt trạm quan trắc trực tiếp.

### **1.9. Bản đồ Folium TPHCM**

In [ ]:
# 1. Dữ liệu tọa độ các trạm quan trắc thực tế tại TP.HCM
data_folium = {
    'Folium': [
        'Lãnh sự quán Mỹ (Quận 1)',
        'RMIT University (Quận 7)',
        'Thảo Điền (Quận 2)',
        'Đại học Bách Khoa (Quận 10)',
        'Khu công nghệ cao (Thủ Đức)'
    ],
    'Lat': [10.7828, 10.7293, 10.8037, 10.7734, 10.8521],
    'Lon': [106.7003, 106.6943, 106.7371, 106.6606, 106.7974]
}
df_folium = pd.DataFrame(data_folium)

# 2. Định nghĩa vùng giới hạn cho TP.HCM
# [[Vĩ độ Nam, Kinh độ Tây], [Vĩ độ Bắc, Kinh độ Đông]]
HCM_bounds = [[10.3500, 106.3500], [11.1500, 107.0500]]

# 3. Khởi tạo bản đồ
HCM_map = folium.Map(
    location=[10.82302, 106.62965], # Trung tâm TP.HCM
    zoom_start=12,
    tiles='CartoDB positron',
    min_zoom=11,
    max_zoom=16,
    max_bounds=True,
    bounds=HCM_bounds
)

# 4. Đánh dấu các trạm
for idx, row in df_folium.iterrows():
    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=8,
        popup=folium.Popup(f"<b>Trạm quan trắc:</b> {row['Folium']}", max_width=250),
        tooltip="Nhấn để xem chi tiết",
        color='#3186cc',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.8
    ).add_to(HCM_map)

# Hiển thị bản đồ
HCM_map

Dựa vào **"Bản đồ Folium (Trạm quan trắc) TP.HCM"** ta nhận thấy:
- **Tính đại diện của mẫu dữ liệu:**

**Quận 1 (Lãnh sự quán), Quận 10 (Bách Khoa):** Trung tâm thành phố, đại diện cho khu vực chịu ảnh hưởng bởi kẹt xe, khói xả từ xe máy trực tiếp, hiệu ứng nhà kính từ các tòa nhà cao tầng và thiếu gió.

**Quận 7 (RMIT), Quận 2 (Thảo Điền):** Khu dân cư cao cấp, đại diện cho khu vực có mật độ cây xanh cao hơn, gần sông lớn (sông Sài Gòn, sông Cảng), gió thoáng. Nếu AQI ở đây mà cao, chứng tỏ ô nhiễm đã bao trùm toàn thành phố chứ không chỉ là ô nhiễm cục bộ do kẹt xe.

**Thủ Đức (Khu CNC):** Cửa ngõ thành phố, đại diện cho nguồn ô nhiễm từ xe tải hạng nặng, container và khí thải công nghiệp.

> **Điểm mù của dữ liệu:** Bản đồ đang thiếu các trạm quan trắc từ các huyện ngoại thành rộng lớn như Bình Chánh, Hóc Môn, Củ Chi, hay khu vực nhiều khu công nghiệp như Bình Tân.

### **1.10. Tổng kết**

#### **Các phát hiện chính**

1. **Chất lượng không khí TPHCM ở mức trung bình** - AQI trung bình 81.6,
   phần lớn thời gian (76.3%) ở mức Moderate. Gần 1/5 số ngày (18.6%)
   ảnh hưởng đến nhóm nhạy cảm.

2. **Tính mùa vụ rõ ràng** - Mùa khô (T11-T4) AQI cao hơn mùa mưa (T5-T10)
   do thiếu mưa rửa trôi bụi và hiện tượng nghịch nhiệt. Biên độ dao động
   khoảng ±30-45 đơn vị AQI theo mùa.

3. **Xu hướng đang xấu dần** - Từ đầu 2024 đến giữa 2025, đường Trend
   tăng liên tục từ ~70 lên >90. Năm 2025 là năm ô nhiễm nhất trong
   toàn bộ chuỗi dữ liệu.

4. **PM2.5 là yếu tố chi phối AQI** - Tương quan r≈0.92,
   cao hơn hẳn các chất khác. Mọi nỗ lực cải thiện không khí cần
   ưu tiên kiểm soát PM2.5.

5. **Nhiễu ngẫu nhiên lớn** - Residual có spike đến ±40 đơn vị,
   phản ánh các sự kiện cục bộ khó dự đoán.

#### **Định hướng Feature Engineering (cho Notebook 02)**

- Mã hóa tính mùa vụ (Tuần hoàn sin/cos)
- Tạo `is_dry_season` (T11-T4 = 1) vì biên độ mùa vụ lớn
- Lag features `pm2_5_lag1`, `us_aqi_lag1` vì PM2.5 tương quan cao nhất
- Rolling mean 7 ngày (hoặc 30 ngày) để nắm bắt xu hướng dài hạn